In [ ]:
"""

See description in BestAddressAnomalies.md

TODO:
- splitter en plusieurs tables (addresses - streets - munic - postalinfo)
- vibecoder une interface
- projection offset: documenter
- street name incoherence: créer region_streets et connected_streets sur base du streetname concaténé, une seule fois. Faire uniquement les calculs de distance par lg

"""

import os
import urllib
import pickle
import warnings

import re
from functools import reduce

import difflib
import requests

import jellyfish


import pandas as pd
import numpy as np

from IPython.display import display

import geopandas as gpd
import shapely
from shapely.geometry import Point, LineString
import shapelysmooth
import contextily as ctx
import xyzservices

import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import ListedColormap

from tqdm.auto import tqdm, trange
tqdm.pandas()

osm_crs = 'epsg:4326'

# Main code

In [ ]:
region_name = "bru"
# region_name = "vlg"
# region_name = "wal"

# region_name = "bru+vlg"


In [ ]:
municipality_id_prefix = None
# municipality_id_prefix = "21018" # Woluwé-Saint-Lambert
# municipality_id_prefix = "21019" # Woluwé-Saint-Pierre
# municipality_id_prefix = "21015" # Schaerbeek
# municipality_id_prefix = "21005" # Etterbeek
# municipality_id_prefix = "21006" # Evere
# municipality_id_prefix = "23094" # Zaventem
# municipality_id_prefix = "23099" # Kraainem
# municipality_id_prefix = "25"
# municipality_id_prefix = "25112" # Wavre
# municipality_id_prefix = "93088" # Walcourt
# municipality_id_prefix = "21016" # Uccle

# municipality_id_prefix = "6"  # Province de Liège
# municipality_id_prefix = "2"  # Brabant flamand/wallon

# • 10000 : Province d'Anvers
# • 30000 : Province de Flandre-Occidentale
# • 40000 : Province de Flandre-Orientale
# • 50000 : Province de Hainaut
# • 60000 :
# • 70000 : Province de Limbourg
# • 80000 : Province de Luxembourg
# • 90000 : Province de Namur

municipality_ids = None
# municipality_ids = ["21018", "23094"]

In [ ]:
warnings.filterwarnings("ignore", category=shapely.errors.ShapelyDeprecationWarning)

## Functions

In [ ]:


def download_if_nexist(url, filename):
    """
    If the (local) file <filename> does not exists, download it from <url>

    Parameters
    ----------
    url: str
       url to fetch
    filename: str
       local file to save

    Returns
    -------

    None
    """
    if not os.path.isfile(filename):
        with urllib.request.urlopen(url) as response:
            with open(filename, "wb") as f:
                f.write(response.read())

In [ ]:


def set_optimal_limits(ax, df):
    """
    Adapt xlim/ylim to a GeoDataFrame point plot to avoid plot to be too wide when
    points are horizontally aligned, and to narrow when points are vertically aligned

    Usage :

    ax = df.plot()
    set_optimal_limits(ax, df)

    Parameters
    ----------
    ax: AxesSubplot
       plot to resize
    df: GeoDataFrame
       data to be plotted

    Returns
    -------
        None
    """

    plot_ratio = 1.5    # optimal ratio between "one horizontal degree" and "one vertical degree". It depends of the CRS.

    minimal_width = 600

    margins = 1.1  # Avoid having dots on edges of the plot

    # Compute dimension of the data
    xmin, ymin, xmax, ymax = df.total_bounds
    height = ymax - ymin
    width = xmax - xmin

    opt_height = max(height, width / plot_ratio, minimal_width / plot_ratio)
    opt_width = max(width, height*plot_ratio, minimal_width)

    # If plot is too narrow, increase xmin. If plot is too wide, increase ylim

    if opt_height > height:
        ymid = (ymax + ymin) / 2
        mid_height = opt_height * margins / 2
        ax.set_ylim(ymid - mid_height, ymid + mid_height)
    if opt_width > width:
        xmid = (xmax + xmin) / 2
        mid_width = opt_width * margins / 2
        ax.set_xlim(xmid - mid_width, xmid + mid_width)

In [ ]:


def add_basemap(ax, zoom='auto'):
    """
    Add a basemap on a plot. Tries first default (Stamen) basemap. If errors, tries OpenStreetMap.Mapnik

    Parameters
    ----------
    ax: matplotlib axes

    Returns
    -------
        None
    """

    try:
        ctx.add_basemap(ax, source=basemap_source, zoom=zoom, headers={'User-Agent': 'SmalsResearch'})
    except requests.HTTPError as e:
        print("HTTP error: ", e)
        try:
            ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, zoom=zoom, headers={'User-Agent': 'SmalsResearch'})
        except requests.HTTPError as e:
            print("HTTP error: ", e)
            print("No basemap available")
    except ValueError as e:
        print("Value error...")
        print(e)
    except NameError:
        print("contextily not installed")

    ax.axis(False)

In [ ]:


def null_jaro(str1, str2):
    """Calculate Jaro-Winkler similarity between two strings, handling null values.

    Args:
        str1 (str): A string to compare, possibly null value
        str2 (str): A string to compare, possibly null value

    Returns:
        float: The Jaro-Winkler similarity between the two strings, or pd.NA if either string is null.
    """

    if pd.isnull(str1) or pd.isnull(str2):
        return pd.NA

    return jellyfish.jaro_winkler_similarity(str1, str2)  # pylint: disable=E1101

In [ ]:


def line_sinuosity(geom):
    """
    Calculate the sinuosity of a line geometry, i.e. the ratio between the distance of the geometry, and the straight distance between the start and the end of the geometry

    Parameters
    ----------
    geom: shapely.geometry.LineString
        The line geometry for which to calculate sinuosity.

    Returns
    -------
    float
        The sinuosity of the line geometry.
    """

    assert geom.geom_type == "LineString", geom.geom_type
    length = geom.length
    start_pt = geom.interpolate(0)
    end_pt = geom.interpolate(1, normalized=True)
    straight_dist = start_pt.distance(end_pt)
    if straight_dist == 0.0:
        if length == 0.0:
            return 0.0
        return float("inf")
    return length / straight_dist

In [ ]:


def sliding_sinuosity(street_side, windows_size=5):
    """Calculate the sliding sinuosity of a street side.

    Parameters
    ----------
    street_side: pandas.DataFrame
        A DataFrame containing the geometry of the street side.
    windows_size: int, default=5
        The size of the window for calculating local sinuosity.

    Returns
    -------
    float
        The mean local sinuosity of the street side.
    """

    if street_side.geometry.nunique() < 3:
        return pd.NA
    geometry = street_side.geometry
    local_sinuosity = [line_sinuosity(LineString(geometry.iloc[i:i+windows_size].reset_index(drop=True))) for i in range(0, max(1, geometry.shape[0] - windows_size))]

    return np.mean(local_sinuosity)

In [ ]:


def bloc_sinuosity(street_side):
    """Calculate the sinuosity of a street side."""
    if street_side.geometry.nunique() < 3:
        return pd.NA
    street_side_linestring = LineString(street_side.reset_index().geometry)

    return line_sinuosity(street_side_linestring)

In [ ]:


def bloc_length(street_side):
    """Calculate the length of a street side."""
    if street_side.geometry.nunique() < 3:
        return pd.NA
    street_side_linestring = LineString(street_side.reset_index().geometry)

    return street_side_linestring.length

In [ ]:


def get_max_delta_ratio(street_bloc):
    """
    Get the maximum delta ratio between house numbers in a street block.
    Delta ratio is defined as the absolute difference between two house numbers (considering only numeric part) divided by the distance between their geometries.

    Parameters
    ----------
    street_bloc: pandas.DataFrame
        A DataFrame containing the house numbers and geometries of the street block.

    Returns
    -------
    tuple
        A tuple containing the maximum delta ratio and the corresponding house number pair.
    """

    if street_bloc.house_number_num.max() - street_bloc.house_number_num.min() < 20:
        return 0, "-"

    b1 = street_bloc.reset_index()[["house_number_num", "house_number", "geometry", "index"]].assign(mg=1)
    b2 = street_bloc.reset_index()[["house_number_num", "house_number", "geometry", "index"]].assign(mg=1)
    dot_prod = b1.merge(b2, on="mg")
    dot_prod = dot_prod[dot_prod.index_x < dot_prod.index_y]

    dot_prod["delta_num"] = (dot_prod.house_number_num_x - dot_prod.house_number_num_y).abs()

    dot_prod = dot_prod[dot_prod["delta_num"] > 10]

    dot_prod["distance"] = gpd.GeoSeries(dot_prod.geometry_x).distance(gpd.GeoSeries(dot_prod.geometry_y))
    dot_prod["delta_ratio"] = dot_prod.delta_num / dot_prod["distance"]

    dot_prod = dot_prod[dot_prod.delta_ratio < np.inf]

    if dot_prod.shape[0] == 0:
        return 0, "-"

    id_max = dot_prod.delta_ratio.idxmax()

    rec_max = dot_prod.loc[id_max]

    return (rec_max.delta_ratio, f"{rec_max.house_number_x} -> {rec_max.house_number_y}")

In [ ]:


def get_street_bloc(region, streetname, postcode, parity):
    """ extract a street block from a region based on street name, postcode, and parity of house numbers."""
    street_bloc = region[(region.streetname == streetname) & (region.postcode == postcode) & (region.house_number_num.mod(2) == parity)].copy()

    street_bloc = street_bloc.sort_values(["house_number_num", "house_number"])

    street_bloc = street_bloc.drop_duplicates(["geometry", "house_number"])

    return street_bloc

In [ ]:


def plot_street_bloc(street_bloc, title=None, ax=None, house_number=None):
    """Plot a street bloc

    Args:
        street_bloc (pd.DataFrame): a part of "region" dataframe, containing a street bloc
        title (str, optional): The title for the plot. Defaults to None.
        ax (matplotlib.axes.Axes, optional): The axes object to plot on. Defaults to None.
    """
    ax = street_bloc.plot(column="house_number_num", figsize=(10, 10), alpha=0.7, ax=ax, zorder=10)

    ax.plot(street_bloc.geometry.x.values, street_bloc.geometry.y.values, alpha=0.5)
    ax.set_title(title)

    for _, row in street_bloc.iterrows():
        ax.annotate(text=row["house_number"],
                    xy=(row["geometry"].x, row["geometry"].y),
                    fontsize="x-small",
                    xytext=(5, 5), textcoords='offset pixels'
                    )

    if house_number is not None:
        if isinstance(house_number, str):
            house_number = [house_number]
        street_bloc_sel = street_bloc[street_bloc.house_number.isin(house_number)]
        if not street_bloc_sel.empty:
            street_bloc_sel.plot(ax=ax, zorder=12, color="red")

        for _, row in street_bloc_sel.iterrows():
            ax.annotate(text=row["house_number"],
                        xy=(row["geometry"].x, row["geometry"].y),
                        fontsize="x-small",
                        xytext=(5, 5), textcoords='offset pixels',
                        color="red",
                        zorder=12
                        )

    set_optimal_limits(ax, street_bloc)

    add_basemap(ax)


In [ ]:


def plot_street_bloc_plotly(street_bloc, house_number=None):
    """ Plot a street bloc using Plotly. Cf 'plot_street_bloc'"""
    street_bloc_osm = street_bloc.to_crs(osm_crs)

    fig = go.Figure(go.Scattermap(mode="markers+lines",
                                  lat=street_bloc_osm.geometry.y,
                                  lon=street_bloc_osm.geometry.x,
                                  marker=dict(color=street_bloc_osm['house_number_num'],
                                              size=10),
                                  hovertext=street_bloc_osm[["house_number", "streetname", "postcode", "postname", "municipality"]]))

    if house_number is not None:
        street_bloc_sel = street_bloc_osm[street_bloc_osm.house_number == house_number]
        # display(street_bloc_sel)
        fig.add_trace(go.Scattermap(mode="markers",
                                    lat=street_bloc_sel.geometry.y,
                                    lon=street_bloc_sel.geometry.x,
                                    marker=dict(color="red",
                                                symbol="circle-stroked",
                                                size=15),
                                    hovertext=street_bloc_sel[["house_number", "streetname", "postcode", "postname", "municipality"]]
                                    ))

    fig.update_layout(
        map={
            'center': {'lon': street_bloc_osm.geometry.x.median(),
                       'lat': street_bloc_osm.geometry.y.median()},
            'style': "open-street-map",
            'zoom': 14,
               },
        width=1200, height=800,

    )

    return fig



In [ ]:

def plot_zip_mismatch_plotly(zip_mismatches, zipcodes_boundaries, zipcode):
    """Plot zip code mismatches

    Args:
        zip_mismatches (pd.DataFrame): DataFrame containing zip code mismatches
        zipcodes_boundaries (gpd.GeoDataFrame): GeoDataFrame containing zip code boundaries
        zipcode (str): The zip code to plot
    """
    mism = zip_mismatches[zip_mismatches.postcode == zipcode].rename(columns={"geometry_left": "geometry"}).set_geometry("geometry")
    nis = ";".join(mism.municipality_id.unique())

    zip_bnd = zipcodes_boundaries[zipcodes_boundaries.zipcode == zipcode].copy()
    zip_bnd.geometry = zip_bnd.boundary

    zip_bnd = zip_bnd.set_geometry("geometry")

    zip_bnd_centroid = {'lon': zip_bnd.geometry.centroid.to_crs(osm_crs).x.iloc[0],
                        'lat': zip_bnd.geometry.centroid.to_crs(osm_crs).y.iloc[0]}

    mism = mism.to_crs(osm_crs)
    zip_bnd = zip_bnd.to_crs(osm_crs)

    zipcodes = mism["zipcode"].dropna().unique()
    colors = px.colors.qualitative.Plotly
    color_map = {
        cat: colors[i % len(colors)]
        for i, cat in enumerate(zipcodes)
    }

    fig = go.Figure()

    for zc in zipcodes:
        mism_zip = mism[mism["zipcode"] == zc]

        fig.add_trace(go.Scattermap(mode="markers",
                                    lat=list(mism_zip.geometry.y),
                                    lon=list(mism_zip.geometry.x),
                                    marker={"size": 10, "color": color_map[zc]},
                                    fillcolor=color_map[zc],
                                    text=mism_zip.apply(lambda x: f"{x.house_number} {x.streetname}, best:{x.postcode}/ bPost:{x.zipcode}) {x.postname}, {x.municipality}", axis=1),
                                    name=f"bPost: {zc}"
                                    ))

    # Add the zip code boundary as a line trace

    lats, lons = [], []

    for geom in zip_bnd.geometry:
        lines = geom.geoms if geom.geom_type == "MultiLineString" else [geom]

        for line in lines:
            x, y = line.xy
            lons.extend(list(x) + [None])
            lats.extend(list(y) + [None])

    fig.add_trace(go.Scattermap(mode="lines",
                                lat=lats,
                                lon=lons,
                                line={"width": 2, "color": "black"},
                                name=f"bPost: {zipcode}"
                                ))
    fig.update_layout(
        map={
            'center': zip_bnd_centroid,
            'style': "open-street-map",
            'zoom': 12,
               },
        width=1200, height=800,
        title=f"Zipcode mismatches for BeSt zipcode={zipcode} / NIS={nis}"
    )

    # fig.show()
    return fig


In [ ]:


def make_table(street_bloc, title=None):
    """Plot a table with street addresses info

    Args:
        street_bloc (pd.DataFrame): a part of "region" dataframe, containing a street bloc
        title (str, optional): The title for the table. Defaults to None.

    Returns:
        matplotlib.figure.Figure: a Matplotlib figure containing the table.
    """
    street_bloc = street_bloc.reset_index()

    street_bloc["dist_to_prev"] = street_bloc.distance(street_bloc.shift(1)).fillna(0).astype(int)

    if street_bloc.shape[0] > 20:
        df1 = street_bloc[["house_number", "address_id", "geometry", "dist_to_prev"]].head(10)
        df2 = street_bloc[["house_number", "address_id", "geometry", "dist_to_prev"]].tail(10)
        df3 = pd.DataFrame(columns=df1.columns,
                           index=["..."],
                           data=[["..."]*df1.shape[1]])

        df = pd.concat([df1, df3, df2])
    else:
        df = street_bloc[["house_number", "address_id", "geometry", "dist_to_prev"]].copy()

    df["address_id"] = df["address_id"].apply(";".join)
    for fld in ["address_id"]:
        df[fld] = df[fld].str[0:30]

    fig, ax = plt.subplots(figsize=(10, .3 * df.shape[0]))
    ax.axis('tight')
    ax.axis('off')

    ax.table(cellText=df.values,
             rowLabels=df.index,
             colLabels=df.columns,
             rowColours=['lightblue']*len(df),
             colColours=['lightblue']*len(df.columns),
             loc='center')
    plt.title(title)
    return fig


In [ ]:


def make_metric_table(rec, metrics, ax=None):
    """Plot a table with metrics info

    Args:
        rec (dict): A dictionary containing the metrics data
        metrics (list): A list of metric names to display
        ax (matplotlib.axes.Axes, optional): The axes object to plot on. Defaults to None.
    """
    metrics_table = pd.DataFrame({m: {
        "metric": f"{rec[m]:.2f}",
        "house_number": rec[f"{m}_house_number"] if f"{m}_house_number" in rec else "",
        "ranking": "" if pd.isnull(rec[f"{m}_ranking"]) else int(rec[f"{m}_ranking"]),
    } for m in metrics})

    if ax is None:
        _, ax = plt.subplots(figsize=(10, 1))
    ax.axis('tight')
    ax.axis('off')

    ax.table(cellText=metrics_table.values,
             rowLabels=metrics_table.index,
             colLabels=metrics_table.columns,
             rowColours=['lightblue']*len(metrics_table),
             colColours=['lightblue']*len(metrics_table.columns),
             loc='center')


In [ ]:


def plot_building_boxes(boxes, full_region, with_explore=False):
    """Plot building boxes with optional exploration mode."""
    r = boxes.reset_index().iloc[0]
    title = f"{r.streetname} ({r.street_id}), {r.house_number}\n{r.postcode}/{r.municipality_id} {r.municipality} ({int(r.mrr_length)} m x {int(r.mrr_width)} m)"

    if with_explore:
        m = boxes.explore()
        return boxes.set_geometry("min_rot_rect").explore(m=m,  color="red")

    ax = boxes.plot()
    boxes.set_geometry("min_rot_rect").plot(ax=ax, alpha=0.2, color="red")
    ax.set_title(title)

    set_optimal_limits(ax, boxes)
    add_basemap(ax)

    all_boxes = boxes.reset_index()[boxes.index.names].merge(full_region)

    for _, row in all_boxes.fillna("/").iterrows():
        ax.annotate(text=row["box_number"], xy=(row["geometry"].x, row["geometry"].y))

In [ ]:


def get_min_rot_rect_size(min_rot_rect):
    """ Get the length and width of a minimum rotated rectangle geometry."""
    if isinstance(min_rot_rect, Point):
        return 0, 0

    if isinstance(min_rot_rect, LineString):
        return min_rot_rect.length, 0
    x, y = min_rot_rect.exterior.coords.xy

    # get length of bounding box edges
    edge_length = (Point(x[0], y[0]).distance(Point(x[1], y[1])), Point(x[1], y[1]).distance(Point(x[2], y[2])))

    # get length of polygon as the longest edge of the bounding box
    length = max(edge_length)

    # get width of polygon as the shortest edge of the bounding box
    width = min(edge_length)

    return length, width

In [ ]:


def plot_zip_mismatch(zip_mismatches, zipcodes_boundaries, zipcode, pdf=None):
    """Plot zip code mismatches

    Args:
        zip_mismatches (pd.DataFrame): DataFrame containing zip code mismatches
        zipcodes_boundaries (gpd.GeoDataFrame): GeoDataFrame containing zip code boundaries
        zipcode (str): The zip code to plot
    """
    mism = zip_mismatches[zip_mismatches.postcode == zipcode].rename(columns={"geometry_left": "geometry"}).set_geometry("geometry")
    nis = ";".join(mism.municipality_id.unique())
    mism = mism[["geometry", "zipcode"]].dissolve(["zipcode"]).reset_index()

    zip_bnd = zipcodes_boundaries[zipcodes_boundaries.zipcode == zipcode].copy()
    zip_bnd.geometry = zip_bnd.boundary

    mism = pd.concat([zip_bnd, mism])
    ax = mism.plot("zipcode",  categories=mism.zipcode, legend=True,
                   cmap=ListedColormap(["C3", "C0", "C1", "C2", "C4", "C5", "C6", "C7", "C8", "C9"][0:mism.shape[0]]))  # To force zip_bnd to be in red (C3)

    plt.title(f"BeSt zipcode / NIS: {zipcode} / {nis}")
    ax.get_legend().set_title("bPost zipcodes")

    set_optimal_limits(ax, mism)
    add_basemap(ax)
    if pdf is not None:
        pdf.savefig(bbox_inches='tight', dpi=dpi)
    else:
        plt.show()
    plt.close()


In [ ]:


def single_digit_diff(a, b):
    """Return True if two strings differ by a single digit, False otherwise.

    Args:
        a (str): First string to compare
        b (str): Second string to compare

    Returns:
        bool: True if the strings differ by a single digit, False otherwise
    """

    if pd.isnull(a) or pd.isnull(b) or len(a) != len(b):
        return False

    diffs = list(difflib.ndiff([a], [b]))

    if len(diffs) != 4:
        return False

    diff_pos_a = diffs[1].count(" ")
    diff_pos_b = diffs[3].count(" ")

    return a[diff_pos_a-1].isdigit() and b[diff_pos_b-1].isdigit()

In [ ]:


def remove_outliers(street_bloc, threshold=100):
    """Remove outliers from a street bloc based on distance to the median point.
    Step one: create buffer of 'threshold' meters around each point and merge overlapping buffers. If a point is alone in its 'cluster', we drop it
    Strep two: (probably useless)
    Args:
        street_bloc (gpd.GeoDataFrame): GeoDataFrame containing the street bloc
        threshold (float): Buffer distance for identifying outliers
    """

    if street_bloc.shape[0] <= 10:
        return street_bloc

    lstring = street_bloc.geometry.values
    buf = lstring.buffer(threshold)

    buffers = buf.union_all()
    if buffers.geom_type == "Polygon":  # Only one cluster, we keep all data
        return street_bloc

    clusters = gpd.GeoDataFrame({'geometry': buffers.geoms}, crs=street_bloc.crs)
    clusters_mg = gpd.sjoin(street_bloc[["geometry", "address_id"]].set_geometry("geometry"), clusters, how="inner", predicate="intersects")

    # display(clusters_mg)

    vc = clusters_mg.index_right.value_counts()
    # display(vc)

    clusters_mg = clusters_mg[clusters_mg.index_right.isin(vc[vc > 1].index)]

    # step 2 ; usefull? Already skipped at previous step
    clusters_mg["dist_prev"] = clusters_mg.distance(clusters_mg.shift(1))
    clusters_mg["dist_next"] = clusters_mg.distance(clusters_mg.shift(-1))

    dist_thresh = 1000

    clusters_mg = clusters_mg[["geometry"]][(clusters_mg.dist_prev.fillna(dist_thresh) < dist_thresh) & (clusters_mg.dist_next.fillna(dist_thresh) < dist_thresh)]

    return clusters_mg.merge(street_bloc, on="geometry").reset_index(drop=True)

In [ ]:


def get_ordered_line(street_bloc):
    """Get an ordered line from a street bloc based on the principal axis.

    Args:
        street_bloc (gpd.GeoDataFrame): GeoDataFrame containing the street bloc
        """
    points = np.array([[pt.x, pt.y] for pt in street_bloc.geometry])

    # Center the points and find the principal axis via SVD
    centered = points - points.mean(axis=0)
    _, _, vh = np.linalg.svd(centered)
    principal_axis = vh[0]

    # Project points onto the principal axis and sort
    projections = centered @ principal_axis
    ordered_points = points[np.argsort(projections)]

    line = LineString(ordered_points)
    return line


# def simplify_road(lstring):
#     # buf = lstring.buffer(100).simplify(10)
#     # display(buf)
#     # return Centerline(buf, interpolation_distance=100).geometry.simplify(1000)

#     smoothed_line = shapelysmooth.taubin_smooth(lstring) #, n_iter=10, lambda_=0.5, mu=-0.53)
#     return smoothed_line

In [ ]:
def get_simplified_street(street_bloc, threshold=100):
    """Get a simplified version of a street bloc by removing outliers and simplifying the geometry.

    Args:
        street_bloc (gpd.GeoDataFrame): GeoDataFrame containing the street bloc
        threshold (float): Buffer distance for identifying outliers
        """
    street_bloc_filtered = remove_outliers(street_bloc, threshold=threshold)
    if street_bloc_filtered.shape[0] <= 5:
        return None

    lstring = get_ordered_line(street_bloc_filtered)
    # lstring = shapely.LineString(street_bloc_filtered.geometry.values)

    simplified_line = shapelysmooth.taubin_smooth(lstring)

    # Remove duplicated points --> should use remove_repeated_points, but doesn't work on this machine...

    x = pd.DataFrame(simplified_line.xy).T
    x = x.drop_duplicates()
    simplified_line = shapely.LineString(x)

    return simplified_line


In [ ]:


def get_projection_anomalies(street_bloc):
    """Get anomalies in a street bloc based on distance to the smoothed line.

    Args:
        street_bloc (gpd.GeoDataFrame): GeoDataFrame containing the street bloc
        """

    smoothed_line = get_simplified_street(street_bloc)

    if smoothed_line is None:
        return None

    street_bloc["smooth_projected"] = street_bloc.apply(lambda row: smoothed_line.project(Point(row.geometry)), axis=1)

    # If smoothed_line was "backward" (small housenumber close to end of line), we need to reverse and project again

    if street_bloc.smooth_projected[0:5].mean() > street_bloc.smooth_projected[-6:-1].mean():
        smoothed_line = smoothed_line.reverse()
        street_bloc["smooth_projected"] = street_bloc.apply(lambda row: smoothed_line.project(Point(row.geometry)), axis=1)

    corr = street_bloc[["smooth_projected", "house_number_num"]].corr()["smooth_projected"]["house_number_num"]

    street_bloc["smooth_distance"] = street_bloc.apply(lambda row: smoothed_line.distance(Point(row.geometry)), axis=1)
    sinuosity = smoothed_line.length / shapely.distance(smoothed_line.boundary.geoms[0], smoothed_line.boundary.geoms[1])
    # print(f"pearson's corr: {corr}")
    # print("avg dist: ", street_bloc["smooth_distance"].mean())
    # print("sinuosity: ", sinuosity)

    # if corr < .9 and street_bloc["smooth_distance"].mean() > 15:
    #     # smoothed_line is not a good approximation --> ignore street
    #     return None

    street_bloc["projection_offset"] = street_bloc.smooth_projected.shift(1).fillna(0) - street_bloc.smooth_projected

    # display(street_bloc[["house_number", "house_number_num", "smooth_projected", "projection_offset"]])
    street_bloc = street_bloc[["projection_offset", "smooth_distance"]].assign(house_number=street_bloc.house_number.shift(1).fillna('') + " -> " + street_bloc.house_number)

    anomalies = street_bloc[(street_bloc.smooth_distance < 100) & ((street_bloc.projection_offset > 0))]
    anomalies = anomalies.assign(pearson_corr=corr,
                                 avg_dist=street_bloc["smooth_distance"].mean(),
                                 sinuosity=sinuosity)
    # display(anomalies)
    return anomalies[["projection_offset", "house_number", "pearson_corr", "avg_dist", "sinuosity"]]


## Parameters

In [ ]:


root_output_dir = "output/best_anomalies"
data_dir = "data/best_anomalies"
topn = 50
dpi = 150

In [ ]:
basemap_source, base = "https://cartoweb.wmts.ngi.be/1.0.0/topo/default/3857/{z}/{y}/{x}.png", "topo"
# basemap_source, base = "https://cartoweb.wmts.ngi.be/1.0.0/topo/default_bw/3857/{z}/{y}/{x}.png", "grey"
# basemap_source, base = "https://cartoweb.wmts.ngi.be/1.0.0/overlay/default/3857/{z}/{y}/{x}.png", "overlay"

basemap_source = xyzservices.TileProvider(name="IGN tiles (base)", url=basemap_source, attribution="(C) CartoWeb.be")

basemap_source, base = ctx.providers.OpenStreetMap.Mapnik, "osm"

In [ ]:
# os.makedirs(output_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

In [ ]:
case_name = f"{region_name}_{base}"

In [ ]:
if municipality_id_prefix is not None:
    case_name = f"{case_name}_{municipality_id_prefix}"
elif municipality_ids is not None:
    case_name = f"{case_name}_{','.join(municipality_ids)}"

In [ ]:
output_dir = f"{root_output_dir}/{case_name}"
os.makedirs(output_dir, exist_ok=True)

## Data reading

In [ ]:
for reg in region_name.split("+"):
    best_fn = f"{data_dir}/openaddress-be{reg}.zip"
    download_if_nexist(f"https://opendata.bosa.be/download/best/openaddress-be{reg}.zip", best_fn)

In [ ]:
full_region = []
for reg in region_name.split("+"):
    best_fn = f"{data_dir}/openaddress-be{reg}.zip"

    full_region.append(pd.read_csv(best_fn, usecols=["municipality_name_de", "municipality_name_fr", "municipality_name_nl", "municipality_id",
                                                     "streetname_de", "streetname_fr", "streetname_nl", "street_id",
                                                     "postname_fr", "postname_nl",
                                                     "postcode", "house_number", "box_number", "region_code",
                                                     "EPSG:3812_x", "EPSG:3812_y",
                                                     "EPSG:4326_lat", "EPSG:4326_lon",
                                                     "address_id", "status"], dtype=str))

full_region = pd.concat(full_region)
full_region

In [ ]:
# with_parenthesis = full_region[full_region.streetname_fr.fillna(full_region.streetname_nl).fillna("").str.contains("(", regex=False)]
# with_parenthesis

In [ ]:
# with_parenthesis.municipality_name_fr.fillna(with_parenthesis.municipality_name_nl).value_counts() / with_parenthesis.shape[0]

In [ ]:
# with_parenthesis[["street_id", "streetname_nl", "streetname_fr", "municipality_name_nl"]].drop_duplicates()

In [ ]:
# with_parenthesis[["street_id", "streetname_nl", "streetname_fr", "municipality_name_nl", "municipality_name_fr"]].drop_duplicates().municipality_name_fr.value_counts()

In [ ]:
# full_region[full_region.house_number.fillna("").str.contains(" ", regex=False)]

In [ ]:
if municipality_id_prefix is not None:
    full_region = full_region[full_region.municipality_id.str.startswith(municipality_id_prefix)].copy()

    full_region[full_region.status == "current"].drop(columns=["EPSG:3812_x", "EPSG:3812_y", "status"]).to_csv(f"{output_dir}/full_region_{region_name}_{municipality_id_prefix}.csv", index=False)

if municipality_ids is not None:
    full_region = full_region[full_region.municipality_id.isin(municipality_ids)].copy()



In [ ]:
print("Without coordinates : ")
#  WAL: 126357
# full_region[full_region["EPSG:31370_x"] == "0.00000"].sort_values(["postcode", "streetname_fr"])
full_region[full_region["EPSG:3812_x"] == "0.00000"].sort_values(["postcode", "streetname_fr"])

In [ ]:
# full_region = full_region[full_region["EPSG:31370_x"] != "0.00000"].copy()
full_region = full_region[full_region["EPSG:3812_x"] != "0.00000"].copy()

In [ ]:
full_region.status.value_counts()

In [ ]:
full_region = full_region[full_region.status == "current"]

In [ ]:
full_region

In [ ]:
lg_order = ["fr", "nl", "de"] if region_name in ["bru", "wal"] else ["nl", "fr", "de"]


def compact_lg(df, field_prefix, lg_ord):
    """ concatenate fields in different languages """
    fields = [f"{field_prefix}_{lg}" for lg in lg_ord]
    columns = df[[fld for fld in fields if fld in df]]
    return columns.apply(list, axis=1).apply(lambda lst: [it for it in lst if pd.notnull(it)]).apply(" / ".join).replace("", "[na]")


full_region["streetname"] = compact_lg(full_region, "streetname", lg_order)

full_region["municipality"] = compact_lg(full_region, "municipality_name", lg_order)

full_region["postname"] = compact_lg(full_region, "postname", lg_order)


full_region["house_number_num"] = full_region.house_number.str.extract("^([0-9]*)").astype(int, errors="ignore")

full_region

In [ ]:
print("No numerical house number:")
print(full_region[full_region.house_number_num == ""].shape[0])
full_region[full_region.house_number_num == ""]

In [ ]:
full_region = full_region[full_region.house_number_num != ""]
full_region.house_number_num = full_region.house_number_num.astype(int)

In [ ]:
crs = "epsg:3857"

full_region["geometry"] = gpd.points_from_xy(full_region["EPSG:3812_x"], full_region["EPSG:3812_y"])
full_region = gpd.GeoDataFrame(full_region)
full_region = full_region.set_crs("epsg:3812").to_crs(crs)
full_region = full_region.drop(["EPSG:3812_x", "EPSG:3812_y"], axis=1)

In [ ]:
#  Workaround as geometries are not "groupbyable"(/hashable)
full_region_wkb = full_region.assign(geometry_wkb=full_region.geometry.apply(lambda geom: geom.wkb))

gb = full_region_wkb.fillna("[na]").groupby(["streetname", "streetname_fr", "streetname_nl", "streetname_de", "street_id",
                                             "house_number", "house_number_num", "postcode", "postname",
                                             "municipality", "municipality_id", "geometry_wkb"])
region = gb.address_id.progress_apply(list).reset_index()
region = region.merge(full_region_wkb[["geometry_wkb", "geometry"]].drop_duplicates()).drop("geometry_wkb", axis=1)

region = gpd.GeoDataFrame(region)
del full_region_wkb

In [ ]:
region = region.sort_values(["postcode", "streetname", "house_number_num"])

In [ ]:
region

# Box anomalies

In [ ]:
# Several names for the same street id
x = full_region[["street_id", "streetname"]].drop_duplicates()
x[x.street_id.duplicated()]

In [ ]:
# Street_ids in multiple municipalities
x = full_region[["street_id", "municipality_id"]].drop_duplicates()
x[x.street_id.duplicated(keep=False)].sort_values("street_id").merge(full_region[["street_id", "municipality_id", "streetname", "municipality"]].drop_duplicates())

In [ ]:
# Several ids for the same street
x = full_region[["street_id", "municipality_id", "streetname", "postcode"]].drop_duplicates()
x[x.duplicated(subset=["streetname", "municipality_id", "postcode"], keep=False)].sort_values("streetname")

In [ ]:
# Several streets with the same name (and difference street ids) with the same municipality
x = full_region[["street_id", "municipality_id", "streetname"]].drop_duplicates()
x[x.duplicated(subset=["streetname", "municipality_id"], keep=False)].sort_values("streetname")

In [ ]:
# Several postcodes for the same building
x = full_region[["street_id", "municipality_id", "house_number", "postcode"]].drop_duplicates()
x = x[x.duplicated(subset=["street_id", "municipality_id", "house_number"], keep=False)].sort_values("street_id")
x.merge(full_region[["street_id", "municipality_id", "postcode", "streetname", "municipality", "house_number", "box_number", "address_id", "geometry"]].drop_duplicates())

In [ ]:
region_with_boxes = full_region[full_region.duplicated(subset=["municipality_id",  "street_id", "house_number"], keep=False)]

In [ ]:
region_with_boxes

In [ ]:
region_buildings = region_with_boxes[["street_id", "streetname", "municipality_id", "municipality",
                                      "postcode", "house_number", "geometry"]].dissolve(["street_id", "municipality_id", "postcode", "house_number"])
region_buildings

In [ ]:
region_buildings["convex_hull"] = region_buildings.convex_hull
region_buildings

In [ ]:
region_buildings["min_rot_rect"] = region_buildings.convex_hull.apply(lambda g: g.minimum_rotated_rectangle)
region_buildings

In [ ]:
mrr_size = region_buildings.min_rot_rect.progress_apply(get_min_rot_rect_size)
region_buildings[["mrr_length", "mrr_width"]] = pd.concat([mrr_size.rename("mrr_length").str[0], mrr_size.rename("mrr_width").str[1]], axis=1)

In [ ]:
region_buildings = region_buildings.sort_values("mrr_length", ascending=False)
region_buildings

In [ ]:
k = 0
plot_building_boxes(region_buildings.iloc[[k]], full_region, with_explore=True)

In [ ]:
pdf = PdfPages(f"{output_dir}/best_anomalies_{case_name}_box_anomalies.pdf")

region_buildings_sel = region_buildings[(region_buildings.mrr_length > 100) & (region_buildings.mrr_width < 100)]
for k in trange(0, min(2*topn, region_buildings_sel.shape[0])):
    boxes = region_buildings_sel.iloc[[k]]

    plot_building_boxes(boxes, full_region)

    pdf.savefig(bbox_inches='tight', dpi=dpi)
    plt.close()

pdf.close()

In [ ]:
del full_region

# Conflict with bPost

In [ ]:
# https://www.geo.be/catalog/details/9738c7c0-5255-11ea-8895-34e12d0f0423?l=fr
zipcode_boundaries_filename = "data/zipcode_boundaries_shapefile_3812.zip"
download_if_nexist("https://bgu.bpost.be/assets/9738c7c0-5255-11ea-8895-34e12d0f0423_x-shapefile_3812.zip",
                   zipcode_boundaries_filename)

In [ ]:
zipcodes_boundaries = gpd.read_file(f"zip://{zipcode_boundaries_filename}/3812")
zipcodes_boundaries["is_special"] = zipcodes_boundaries.CP_speciau == 1
zipcodes_boundaries = zipcodes_boundaries.rename({"nouveau_PO": "zipcode"}, axis=1)[["zipcode", "is_special", "geometry"]]

zipcodes_boundaries = zipcodes_boundaries.dissolve(["zipcode", "is_special"]).reset_index()

zipcodes_boundaries = zipcodes_boundaries.to_crs(crs)
zipcodes_boundaries

In [ ]:
zipcodes_boundaries["buffer"] = zipcodes_boundaries.buffer(-50)

In [ ]:
region_zipcode = gpd.sjoin(region, zipcodes_boundaries.set_geometry("buffer"))

In [ ]:
zip_mismatches = region_zipcode[region_zipcode.postcode != region_zipcode.zipcode]
zip_mismatches = zip_mismatches.merge(zipcodes_boundaries[["zipcode", "geometry"]].rename(columns={"geometry": "zip_geometry", "zipcode": "postcode"}))
zip_mismatches

In [ ]:
zip_mismatches["dist_to_zip_boundary"] = zip_mismatches.distance(zip_mismatches.zip_geometry)

In [ ]:
zip_mismatches.drop(["geometry_right", "index_right"], axis=1).to_excel(f"{output_dir}/best_anomalies_{case_name}_zip_mismatches.xlsx")

In [ ]:
zip_mismatches.sort_values("dist_to_zip_boundary", ascending=False).head(20)

In [ ]:

pdf = PdfPages(f"{output_dir}/best_anomalies_{case_name}_zip_mismatches.pdf")

for zipcode in tqdm(zip_mismatches.postcode.value_counts().index[0:50]):
    plot_zip_mismatch(zip_mismatches, zipcodes_boundaries, zipcode, pdf=pdf)

pdf.close()

In [ ]:
# plot_zip_mismatch(zip_mismatches, zipcodes_boundaries, "1006")
plot_zip_mismatch_plotly(zip_mismatches, zipcodes_boundaries, "1060")
# plt.show()

In [ ]:
pdf = PdfPages(f"{output_dir}/best_anomalies_{case_name}_zip_mismatches_far_addresses.pdf")

for zipcode in tqdm(zip_mismatches[zip_mismatches.dist_to_zip_boundary > 2000].postcode.unique()):
    plot_zip_mismatch(zip_mismatches, zipcodes_boundaries, zipcode, pdf=pdf)

pdf.close()

# Conflict with NIS code

In [ ]:
download_if_nexist("https://statbel.fgov.be/sites/default/files/files/opendata/Statistische%20sectoren/sh_statbel_statistical_sectors_31370_20200101.shp.zip",
                   "data/stat_sectors_2020.zip")
statistical_sectors = gpd.read_file("zip://data/stat_sectors_2020.zip/sh_statbel_statistical_sectors_20200101.shp")
statistical_sectors["CNIS5_2020"] = statistical_sectors["CNIS5_2020"].astype(str)

In [ ]:
# Group (with "dissolve") sectors per NIS code
nis_boundaries = statistical_sectors[["CNIS5_2020", "T_MUN_FR", "T_MUN_NL", "geometry"]].dissolve(by="CNIS5_2020").reset_index()
nis_boundaries = nis_boundaries.rename({"CNIS5_2020": "niscode"}, axis=1)
nis_boundaries = nis_boundaries.to_crs(crs)

In [ ]:
nis_boundaries["buffer"] = nis_boundaries.buffer(-50)

In [ ]:
region_niscode = gpd.sjoin(region, nis_boundaries.set_geometry("buffer"))

In [ ]:
nis_mismatches = region_niscode[region_niscode.municipality_id != region_niscode.niscode]
nis_mismatches = nis_mismatches.merge(nis_boundaries[["niscode", "geometry"]].rename(columns={"geometry": "nis_geometry", "niscode": "niscode2"}), left_on="municipality_id", right_on="niscode2")
nis_mismatches

In [ ]:
nis_mismatches["dist_to_nis_boundary"] = nis_mismatches.set_geometry("geometry").distance(nis_mismatches.nis_geometry)

In [ ]:
nis_mismatches.drop(["geometry_right", "index_right", "nis_geometry"], axis=1).to_excel(f"{output_dir}/best_anomalies_{case_name}_nis_mismatches.xlsx")

In [ ]:
pdf = PdfPages(f"{output_dir}/best_anomalies_{case_name}_nis_mismatches.pdf")

for niscode in tqdm(nis_mismatches.municipality_id.value_counts().index):

    mism = nis_mismatches[nis_mismatches.municipality_id == niscode].set_geometry("geometry")

    ax = mism.plot("niscode", figsize=(10, 10), alpha=0.8, legend=True)
    plt.title(niscode)
    nis_bnd = nis_boundaries[nis_boundaries.niscode == niscode]
    if nis_bnd.shape[0] > 0:
        ax = nis_bnd.boundary.plot(ax=ax, color="red")
    set_optimal_limits(ax, pd.concat([nis_bnd[["geometry"]], mism[["geometry"]]]))
    add_basemap(ax)

    pdf.savefig(bbox_inches='tight', dpi=dpi)
    plt.close()

pdf.close()

# Street names incoherence

In [ ]:
region_streets_lg = {}
for lg in lg_order:
    region_streets = region[region[f"streetname_{lg}"] != "[na]"][[f"streetname_{lg}", "street_id", "postcode", "geometry", "municipality_id"]]
    region_streets = region_streets.rename(columns={f"streetname_{lg}": "streetname"})
    region_streets = region_streets[["streetname", "street_id", "postcode", "geometry", "municipality_id"]].dissolve(["streetname", "postcode"])
    region_streets["buffer"] = region_streets.buffer(100)
    region_streets = region_streets.reset_index()
    display(region_streets)
    region_streets_lg[lg] = region_streets

In [ ]:
connected_streets = {lg: gpd.sjoin(region_streets_lg[lg], region_streets_lg[lg].set_geometry("buffer")) for lg in lg_order}
connected_streets = {lg: connected_streets[lg][connected_streets[lg]["streetname_left"] < connected_streets[lg]["streetname_right"]] for lg in lg_order}

In [ ]:
# connected_streets[lg_order[0]]

In [ ]:
for lg in lg_order:
    for side in ["right", "left"]:
        connected_streets[lg][[f"split_{side}_1", f"split_{side}_2"]] = connected_streets[lg][f"streetname_{side}"].str.extract("^(Avenue|Rue|Chaussée|Boulevard|Drève|Clos|Square)(.*)$",
                                                                                                                                flags=re.IGNORECASE)

In [ ]:
for lg in lg_order:
    if connected_streets[lg].shape[0] == 0:
        continue
    connected_streets[lg]["jaro"] = connected_streets[lg].progress_apply(lambda row: jellyfish.jaro_winkler_similarity(row["streetname_left"],    # pylint: disable=E1101
                                                                                                                       row["streetname_right"]), axis=1)

    connected_streets[lg]["levenshtein"] = connected_streets[lg].progress_apply(lambda row: jellyfish.damerau_levenshtein_distance(row["streetname_left"],    # pylint: disable=E1101
                                                                                                                                   row["streetname_right"]), axis=1)

    connected_streets[lg]["jaro_split"] = connected_streets[lg].progress_apply(lambda row: null_jaro(row["split_left_2"],
                                                                                                     row["split_right_2"]), axis=1)

    # single_last_letter : True if the only difference is the last letter, and the penultimate letter is a space
    connected_streets[lg]["single_last_letter"] = (connected_streets[lg].streetname_left.str.len() == connected_streets[lg].streetname_right.str.len()) & \
                                                  (connected_streets[lg].streetname_left.str[:-2] == connected_streets[lg].streetname_right.str[:-2]) & \
                                                  (connected_streets[lg].streetname_left.str[-2] == " ")

In [ ]:
for lg in lg_order:
    connected_streets[lg]["single_digit_diff"] = connected_streets[lg].progress_apply(lambda row: single_digit_diff(row["streetname_left"],
                                                                                                                    row["streetname_right"]), axis=1)

In [ ]:
streetname_mismatches = {
    lg: connected_streets[lg][((connected_streets[lg].levenshtein <= 1)
                               | (connected_streets[lg].jaro >= 0.96)
                               | ((connected_streets[lg].jaro_split >= 0.96) &
                                  (connected_streets[lg].split_left_1.str.lower() == connected_streets[lg].split_right_1.str.lower()))) &
                              ~connected_streets[lg].single_last_letter &
                              ~connected_streets[lg].single_digit_diff
                              ] if connected_streets[lg].shape[0] > 0 else pd.DataFrame()
    for lg in lg_order}
# streetname_mismatches

In [ ]:
for lg in lg_order:
    streetname_msmtch = streetname_mismatches[lg]
    if streetname_msmtch.shape[0] == 0:
        continue
    for (case, str_msmtch) in [("same_nis", streetname_msmtch[streetname_msmtch.municipality_id_left == streetname_msmtch.municipality_id_right]),
                               ("diff_nis", streetname_msmtch[streetname_msmtch.municipality_id_left != streetname_msmtch.municipality_id_right]),
                               ]:

        if str_msmtch.shape[0] == 0:
            print("No case for", case)
            continue
        pdf = PdfPages(f"{output_dir}/best_anomalies_{case_name}_close_streetnames_{case}_{lg}.pdf")

        for i, rec in tqdm(str_msmtch.sort_values(["levenshtein", "jaro"], ascending=[True, False]).iterrows(),
                           total=str_msmtch.shape[0]):

            r1 = rec[["streetname_left", "street_id_left", "postcode_left",  "municipality_id_left", "geometry"]].rename({
                        "streetname_left": "streetname",
                        "street_id_left": "street_id",
                        "postcode_left": "postcode",
                        "municipality_id_left": "municipality_id"})
            r2 = rec[["streetname_right", "street_id_right", "postcode_right", "municipality_id_right", "geometry_right"]].rename({
                        "streetname_right": "streetname",
                        "street_id_right": "street_id",
                        "postcode_right": "postcode",
                        "municipality_id_right": "municipality_id",
                        "geometry_right": "geometry"})

            r = gpd.GeoDataFrame([r1, r2])

            r["name"] = r["streetname"] + " (" + r["street_id"] + "), " + r["postcode"] + "/" + r["municipality_id"]
            # r["name"] = f'{r["streetname"]} ({r["street_id"]}), {r["postcode"]} / {r["municipality_id"]}'
            ax = r.plot("name", legend=True)
            set_optimal_limits(ax, r)
            add_basemap(ax)

            pdf.savefig(bbox_inches='tight', dpi=dpi)
            plt.close()

        pdf.close()


# Metrics

## Sinuosity

In [ ]:
sin_par = []
for parity in [0, 1]:
    sin = region[(region.house_number_num.mod(2) == parity)].groupby(["streetname", "street_id", "postcode"])[["geometry"]].progress_apply(bloc_sinuosity)
    sin = sin.sort_values(na_position="first").rename("sinuosity").reset_index()
    sin["parity"] = parity
    display(sin)
    sin_par.append(sin)
sinuosity = pd.concat(sin_par)
sinuosity = sinuosity[(sinuosity.sinuosity.notnull()) & (sinuosity.sinuosity < 10**10)].sort_values("sinuosity", ascending=False).reset_index(drop=True)
sinuosity

In [ ]:
# sinuosity = sinuosity[sinuosity.sinuosity<10**10]
# region

In [ ]:
sw_sin_par = []
for parity in [0, 1]:
    sin = region[(region.house_number_num.mod(2) == parity)].groupby(["streetname", "street_id", "postcode"])[["geometry"]].progress_apply(sliding_sinuosity)
    sin = sin.sort_values(na_position="first").rename("sw_sinuosity").reset_index()
    sin["parity"] = parity
    display(sin)
    sw_sin_par.append(sin)
sw_sinuosity = pd.concat(sw_sin_par)
sw_sinuosity = sw_sinuosity[(sw_sinuosity.sw_sinuosity.notnull()) & (sw_sinuosity.sw_sinuosity < 10**10)].sort_values("sw_sinuosity", ascending=False)
sw_sinuosity = sw_sinuosity.reset_index(drop=True)
sw_sinuosity

## Length

In [ ]:
region_lenghts = []
for parity in [0, 1]:
    region_par = region[(region.house_number_num.mod(2) == parity)].copy()

    length = region_par.groupby(["streetname", "street_id", "postcode"])[["geometry"]].progress_apply(bloc_length)
    length = length.sort_values(na_position="first", ascending=False).rename("length").reset_index()

    region_lenghts.append(length.assign(parity=parity))
length = pd.concat(region_lenghts)
length = length.sort_values("length", ascending=False).reset_index(drop=True)
length

## Distance to previous

In [ ]:
region_pars = []
for parity in [0, 1]:
    region_par = region[(region.house_number_num.mod(2) == parity)].copy()

    region_par["dist_to_prev"] = region_par.distance(region_par.shift(1))
    region_par["hn_prev"] = region_par.house_number.shift(1)

    region_par["dist_to_prev2"] = region_par.distance(region_par.shift(2))
    region_par["is_new_bloc"] = (region_par[["streetname", "postcode"]] != region_par[["streetname", "postcode"]].shift(1)).any(axis=1)
    region_par.dist_to_prev = region_par.dist_to_prev.where(~region_par.is_new_bloc, pd.NA)

    region_par.dist_to_prev2 = region_par.dist_to_prev2.where(~region_par.is_new_bloc, pd.NA)
    region_par.dist_to_prev2 = region_par.dist_to_prev2.where(~region_par.is_new_bloc.shift(1).astype(bool), pd.NA)

    region_pars.append(region_par.assign(parity=parity))
region_pars = pd.concat(region_pars)

In [ ]:
idx_max = region_pars.groupby(["streetname", "street_id", "postcode", "parity"]).dist_to_prev.idxmax()
dist_to_prev = region_pars.loc[idx_max.dropna().values].sort_values("dist_to_prev", ascending=False)
dist_to_prev["house_number"] = dist_to_prev["hn_prev"].fillna("-")+" -> "+dist_to_prev["house_number"].fillna("-")
dist_to_prev = dist_to_prev[["streetname", "street_id", "postcode", "parity", "dist_to_prev", "house_number", "house_number_num"]].reset_index(drop=True)
dist_to_prev

In [ ]:
# dist_to_prev.dist_to_prev.describe(percentiles=[0.5, 0.90, .95, .99, .999])

## Delta dist to prev

In [ ]:
idx_max = region_pars.groupby(["streetname", "street_id", "postcode", "parity"]).dist_to_prev.idxmax()
delta_dist_to_prev = region_pars.loc[idx_max.dropna().values].rename(columns={"dist_to_prev": "max_dist_to_prev"})
delta_dist_to_prev

In [ ]:
delta_dist_to_prev = delta_dist_to_prev.merge(region_pars.groupby(["streetname", "postcode", "parity"]).dist_to_prev.median().rename("median_dist_to_prev").reset_index())
delta_dist_to_prev["median_dist_to_prev"] = delta_dist_to_prev["median_dist_to_prev"].where(delta_dist_to_prev.median_dist_to_prev > 5, 0)

delta_dist_to_prev["delta_dist_to_prev"] = delta_dist_to_prev.max_dist_to_prev / delta_dist_to_prev.median_dist_to_prev

delta_dist_to_prev = delta_dist_to_prev[~delta_dist_to_prev.delta_dist_to_prev.isnull() & (delta_dist_to_prev.delta_dist_to_prev < np.inf)].sort_values("delta_dist_to_prev", ascending=False)
delta_dist_to_prev = delta_dist_to_prev[["streetname", "street_id", "postcode", "parity", "house_number", "delta_dist_to_prev"]].reset_index(drop=True)
delta_dist_to_prev

## prev_to_prev2_ratio

In [ ]:
region_pars["prev_to_prev2_ratio"] = (region_pars.dist_to_prev / region_pars[["dist_to_prev2"]].assign(m=10).max(axis=1, skipna=False))
region_pars["prev_to_prev2_ratio"]

In [ ]:
idx_max = region_pars.groupby(["streetname", "postcode", "parity"]).prev_to_prev2_ratio.idxmax()
prev_to_prev2_ratio = region_pars.loc[idx_max.dropna().values].sort_values("prev_to_prev2_ratio", ascending=False)
prev_to_prev2_ratio = prev_to_prev2_ratio[["streetname", "street_id", "postcode", "parity", "prev_to_prev2_ratio", "house_number"]].reset_index(drop=True)
prev_to_prev2_ratio

In [ ]:
# region_pars.sort_values("prev_to_prev2_ratio", ascending=False)

## Delta ratio

In [ ]:
# To avoid to compare all pairs of addresses in all street, we only consider streets with a sinuosity above 1.5
# (assuming that "straight streets" won't have high delta ratio)

region_sel = region.merge(sinuosity[sinuosity.sinuosity > 1.5][["streetname", "postcode"]])
# region_sel

In [ ]:
delta_par = []
for parity in [0, 1]:
    dlt_par = region_sel[(region_sel.house_number_num.mod(2) == parity)].groupby(["streetname", "street_id", "postcode"])
    dlt_par = dlt_par[["house_number_num", "house_number", "geometry"]].progress_apply(get_max_delta_ratio)
    dlt_par = dlt_par.apply(pd.Series).rename(columns={0: "delta_ratio", 1: "house_number"})
    dlt_par = dlt_par.sort_values("delta_ratio", na_position="first", ascending=False)
    dlt_par["parity"] = parity
    display(dlt_par)
    delta_par.append(dlt_par)
delta_ratio = pd.concat(delta_par)

delta_ratio = delta_ratio.sort_values("delta_ratio", ascending=False).reset_index()
delta_ratio

## Projection offset

In [ ]:
# street_bloc = get_street_bloc(region,
#                               "Avenue du Fort-Jaco / Fort-Jacolaan",
#                               "1180",  # "4845",
#                               1)
# get_projection_anomalies(street_bloc)

In [ ]:


projection_offset = []
for parity in [0, 1]:
    prj_par = region[(region.house_number_num.mod(2) == parity)].groupby(["streetname", "street_id", "postcode"])[["house_number", "house_number_num",
                                                                                                                   "geometry", "address_id"]].progress_apply(get_projection_anomalies)
    prj_par["parity"] = parity
    projection_offset.append(prj_par)
projection_offset = pd.concat(projection_offset)

projection_offset = projection_offset.sort_values("projection_offset", ascending=False).reset_index()

projection_offset


## Consolidate

In [ ]:
metrics = {
    "dist_to_prev":       dist_to_prev,
    "delta_dist_to_prev": delta_dist_to_prev,
    "sinuosity":          sinuosity,
    "sw_sinuosity":       sw_sinuosity,
    "length":             length,
    "delta_ratio":        delta_ratio,
    "prev_to_prev2_ratio":   prev_to_prev2_ratio,
    "projection_offset":  projection_offset
}

In [ ]:
with open(f"{data_dir}/metrics_{case_name}.pkl", "wb") as pkl:
    pickle.dump(metrics, pkl, pickle.HIGHEST_PROTOCOL)

region.to_pickle(f"{data_dir}/data_{case_name}.pkl")

In [ ]:
# Start from here to plot without recomputing all metrics
with open(f"{data_dir}/metrics_{case_name}.pkl", "rb") as pkl:
    metrics = pickle.load(pkl)
region = pd.read_pickle(f"{data_dir}/data_{case_name}.pkl")

In [ ]:
for m in metrics:
    metrics[m] = metrics[m].reset_index(drop=True).copy()
    metrics[m][f"{m}_ranking"] = metrics[m].index+1
    if "house_number" in metrics[m]:
        metrics[m] = metrics[m].rename(columns={"house_number": f"{m}_house_number"})


In [ ]:
glob_metrics = reduce((lambda x, y: x.merge(y, how="outer")), metrics.values())

## Show

In [ ]:
region

In [ ]:
k = 0
metric_name = "dist_to_prev"
metric_name = "sinuosity"
metric_name = "delta_ratio"

metric = metrics[metric_name]
print(metric.iloc[k])
street_bloc = get_street_bloc(region,
                              metric.iloc[k].streetname,
                              metric.iloc[k].postcode,
                              metric.iloc[k].parity)

street_bloc

In [ ]:
street_bloc = get_street_bloc(region,
                              "Avenue du Fort-Jaco / Fort-Jacolaan",
                              "1180",
                              1)
display(street_bloc.head())
plot_street_bloc_plotly(street_bloc, "171A")

In [ ]:
smooth_line = get_simplified_street(street_bloc)
ax = street_bloc.plot(figsize=(10, 18))
if smooth_line is not None:
    gpd.GeoSeries(smooth_line).plot(ax=ax, color="green")

add_basemap(ax)

## PDF

In [ ]:
metric

In [ ]:
thresholds = {
    "dist_to_prev": 1000,
    "length": 5000,
    "delta_ratio": 1.0,
    "delta_dist_to_prev": 50,
    "sinuosity": 10,
    "sw_sinuosity": 10,
    "prev_to_prev2_ratio": 10,
    "projection_offset": 100
}

In [ ]:
# for metric_name in ["projection_offset"]:
for metric_name in metrics:
    metric = metrics[metric_name]
    print(metric_name)
    pdf = PdfPages(f"{output_dir}/best_anomalies_{case_name}_{metric_name}.pdf")

    if metric_name in thresholds:
        metric = metric[metric[metric_name] > thresholds[metric_name]]
    for i, metr in tqdm(metric.iloc[0:topn].iterrows(), total=min(topn, metric.shape[0])):

        street_bloc = get_street_bloc(region,
                                      metr.streetname,
                                      metr.postcode,
                                      metr.parity)
        sb = street_bloc.iloc[0]
        title = f"{sb.streetname} ({sb.street_id}) - parity: {metr['parity']}"
        title += f"\n{sb.postcode}/{sb.municipality_id} - {sb.postname} - {sb.municipality}"
        title += f"\n{metric_name}: {metr[metric_name]:.2f}"
        if f"{metric_name}_house_number" in metr:
            title += f" (hn: {metr[f'{metric_name}_house_number']})"
            hn = metr[f"{metric_name}_house_number"]
            if isinstance(hn, str) and " -> " in hn:
                hn = hn.split(" -> ")
        else:
            hn = None

        make_table(street_bloc, title)
        pdf.savefig(bbox_inches='tight')
        plt.close()
        plot_street_bloc(street_bloc, title, house_number=hn)
        pdf.savefig(bbox_inches='tight', dpi=dpi)
        plt.close()
    pdf.close()

In [ ]:
with pd.ExcelWriter(f"{output_dir}/best_anomalies_{case_name}_metrics.xlsx") as writer:
    for metric_name in metrics:
        metric = metrics[metric_name]

        if metric_name in thresholds:
            metric = metric[metric[metric_name] > thresholds[metric_name]]

        metric.to_excel(writer, sheet_name=metric_name)


### Consolidated pdf

In [ ]:
# glob_metrics[(glob_metrics[[f"{m}_ranking" for m in metrics]] < topn).fillna(False).any(axis=1)]
# m= "delta_ratio"
glob_metrics[(glob_metrics[[f"{m}_ranking" for m in metrics]] < topn).fillna(False).any(axis=1)][[f"{m}_ranking" for m in metrics]]

In [ ]:
glob_metrics_topn = glob_metrics[(glob_metrics[[f"{m}_ranking" for m in metrics]] < topn).fillna(False).any(axis=1)]

glob_metrics_topn = glob_metrics_topn[pd.concat([glob_metrics_topn[m] > thresholds[m] for m in metrics], axis=1).any(axis=1)]

glob_metrics_topn = glob_metrics_topn.sort_values("dist_to_prev", ascending=False)
glob_metrics_topn

In [ ]:
glob_metrics_topn["global_ranking"] = glob_metrics_topn[glob_metrics_topn.columns[glob_metrics_topn.columns.str.endswith("_ranking")]].fillna(100).apply(np.log).sum(axis=1)

In [ ]:
glob_metrics_topn = glob_metrics_topn.sort_values("global_ranking")
glob_metrics_topn

In [ ]:
glob_metrics_topn[glob_metrics_topn.duplicated(subset=["streetname", "postcode", "parity"], keep=False)]

In [ ]:
# Keep only one result per street/postcode/parity
glob_metrics_topn = glob_metrics_topn.drop_duplicates(subset=["streetname", "postcode", "parity"])
glob_metrics_topn

In [ ]:
pdf = PdfPages(f"{output_dir}/best_anomalies_{case_name}_consolidated.pdf")

for i, metr in tqdm(glob_metrics_topn.iterrows(), total=glob_metrics_topn.shape[0]):
    street_bloc = get_street_bloc(region,
                                  metr.streetname,
                                  metr.postcode,
                                  metr.parity)

    sb = street_bloc.iloc[0]
    title = f"{sb.streetname} - {sb.postcode}/{sb.municipality_id} - {sb.postname} - {sb.municipality} - parity: {metr['parity']} "

    make_table(street_bloc, title)
    pdf.savefig(bbox_inches='tight')
    plt.close()

    fig, ax = plt.subplots(nrows=2, figsize=(10, 10), gridspec_kw={'height_ratios': [0.85, 0.15]})
    plot_street_bloc(street_bloc, title, ax=ax[0])
    make_metric_table(metr, metrics, ax=ax[1])

    pdf.savefig(bbox_inches='tight', dpi=dpi)
    plt.close()
pdf.close()

# Street density

In [ ]:
# To run this, run this notebook from start, up to "# Box anomalies" (not included)

assert False
full_region = pd.DataFrame()

In [ ]:
housenumber_density = full_region.fillna("").groupby(["street_id", "streetname_fr", "streetname_nl"]).house_number_num.agg(["max", "min", "nunique"])
housenumber_density["density"] = housenumber_density["nunique"]/(housenumber_density["max"] - housenumber_density["min"]+1)
housenumber_density

In [ ]:
housenumber_density.sort_values("density").iloc[120:180]
housenumber_density[housenumber_density.density.between(0.2, 0.4)].sort_values("density")

In [ ]:
x = housenumber_density.reset_index().fillna("").merge(full_region[["street_id", "postcode", "municipality_id", "municipality_name_fr", "municipality_name_nl"]].drop_duplicates().fillna(""))
x.groupby(["municipality_id", "municipality_name_fr", "municipality_name_nl"]).density.mean().sort_values()

In [ ]:
full_region[full_region.street_id == "112692"].sort_values("house_number_num")
